In [ ]:
import argparse
import json
from pathlib import Path
import random
import schedulefree

import numpy as np
import torch
import wandb

import config
from data.utils import DataReader, get_dataset
import distributed
from models.utils import get_model
from optim.base import train
from optim.utils import cos_inf_schedule, wsd_schedule
from optim.cage import CAGE


/home/local/QCRI/kisufaj/miniconda3/envs/qwen/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/local/QCRI/kisufaj/miniconda3/envs/qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def args_from_list(rem_args, config_format="base"):
    """
    Parse args using the repo's config system, but from an explicit list.
    This avoids Jupyter's ipykernel argv pollution.
    """
    parser = argparse.ArgumentParser(allow_abbrev=False, prog="main.py")
    parser.add_argument("--config_format", default=config_format, choices=config.registered_formats())

    # Create a namespace with config_format set, then parse the rest using config system
    base_args = parser.parse_args(["--config_format", config_format])
    return config.parse_args_with_format(
        format=base_args.config_format,
        base_parser=parser,
        args=rem_args,
        namespace=base_args
    )

# 3) Translate your "train.sh style" intent into real main.py args
rem_args = [
    "--experiment-name", "nb_debug_50M",
    "--device", "cuda:7",
    "--dataset", "c4",
    "--datasets-dir", "/image-generation/kisufaj/optimization/cage/CAGE/datasets",
    "--model", "llama",

    "--opt", "adamw",
    "--seed", "0",
    "--data-seed", "0",

    "--batch-size", "64",
    "--acc-steps", "8",
    "--sequence-length", "512",

    # 50M mapping (from your bash function)
    "--n-layer", "7",
    "--n-embd", "768",
    "--n-head", "6",
    "--lr", "0.0012",

    # training length (set something small for notebook debugging)
    "--iterations", "1000",
    "--warmup-steps", "100",

    # CAGE settings
    "--use-cage", "True",
    "--cage-lambda", "0.5",
    "--cage-silence-ratio", "0.5",
    "--cage-schedule", "linear_ramp",
    "--cage-track-stats", "True",

    # Quantization
    "--w-quant", "HadamardMSEQuantizer",
    "--w-quant-kwargs", '{"bits": 4}',
    "--a-quant", "HadamardMSEQuantizer",
    "--a-quant-kwargs", '{"bits": 4}',
    "--auto-resume", False

    # Turn off wandb for notebook experimentation unless you explicitly want it
    # "--wandb",
    # "--wandb-project", "CAGE-QCRI",
    # "--wandb-run-prefix", "NOTEBOOK",
]

args = args_from_list(rem_args, config_format="base")

In [ ]:
def get_exp_name(args, distributed_backend):
    """Returns the name of the experiment, used for saving models and wandb."""
    if args.experiment_name is not None:
        return args.experiment_name

    rank = distributed_backend.rank

    exp_name = (
        f"{args.dataset}_{args.model}_nlayers{args.n_layer}"
        f"_nhead{args.n_head}_lr{args.lr}"
        f"_sched_{args.scheduler}_warmup{args.warmup_steps}"
        f"_decay_{args.decay_type}_{args.wsd_fract_decay}"
        f"_iter{args.iterations}"
        f"_bs{args.batch_size}x{args.acc_steps}_ws{args.world_size}"
    )
    # for mup
    if args.model == "mup_noam":
        exp_name = (
            f"{args.dataset}_{args.model}"
            f"_opt{args.opt}"
            f"_nlayers{args.n_layer}"
            # f"_nhead{args.n_head}"
            f"_lr{args.lr}"
            f"_sched_{args.scheduler}"
            f"_decay_{args.decay_type}"
            # f"_warmup{args.warmup_steps}"
            f"_iter{args.iterations}"
            f"_init{args.init_std}_sce{args.scale_emb}"
            f"_scd{args.scale_depth}"
            # f"_bs{args.batch_size}x{args.acc_steps}_ws{args.world_size}"
        )
    if args.wandb_run_prefix != "none":
        exp_name = args.wandb_run_prefix + "_" + exp_name
    exp_name += f"_seed{args.seed - rank}"
    exp_name += f"_data_seed{args.data_seed}"

    if args.weight_average:
        exp_name += f"_WA"
    if args.opt == "SFAdamW":
        exp_name += f"_beta1_{args.beta1}"
        exp_name += f"_beta2_{args.beta2}"
    return exp_name

In [ ]:
def get_data_readers(args, verbose=True):
    data_srcs = get_dataset(args)
    train_reader = DataReader(
        data_src=data_srcs["train"],
        batch_size=args.batch_size,
        sequence_length=args.sequence_length,
        seed=args.data_seed,
        with_replacement=False,
        auto_shard=True,
        keep_in_ram=args.data_in_ram,
    )
    val_reader = DataReader(
        data_src=data_srcs["val"],
        batch_size=args.batch_size,
        sequence_length=args.sequence_length,
        seed=args.data_seed,
        with_replacement=False,
        auto_shard=False,  # NOTE Identical Per Rank
        keep_in_ram=args.data_in_ram,
    )

    if verbose:
        print(f"Num training tokens: {train_reader.num_tokens}")
        print(f"Num validation tokens: {val_reader.num_tokens}")

    return {
        "train": train_reader,
        "val": val_reader,
    }

In [ ]:
torch.serialization.add_safe_globals([np._core.multiarray._reconstruct, np.ndarray, np.dtype])

distributed_backend = distributed.make_backend_from_args(args)
args = distributed_backend.get_adjusted_args_for_process(args)
args.world_size = distributed_backend.get_world_size()

if args.full_eval_at is None:
    args.full_eval_at = []

# NOTE args.seed is offset per worker in get_adjusted_args_for_process
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.manual_seed(args.seed)
random.seed(args.seed)
np.random.seed(args.seed)
if "cuda" in args.device:
    torch.cuda.set_device(torch.device(args.device))
# torch.use_deterministic_algorithms(True)  # CUBLAS_WORKSPACE_CONFIG=:4096:8

exp_name = get_exp_name(args, distributed_backend)
exp_dir = Path(args.results_base_folder) / exp_name
if distributed_backend.is_master_process() and args.wandb:
    wandb.init(
        project=args.wandb_project,
        name=exp_name,
        config=vars(args),
    )
    wandb.define_metric("iter")
    wandb.define_metric("train/*", step_metric="iter")
    wandb.define_metric("val/*", step_metric="iter")
    wandb.define_metric("lr", step_metric="iter")
    # wandb.define_metric("exp_avg_sq/mean_hist", step_metric="iter")
print(f"Starting Experiment: {exp_name}")
print(f"Experiment Directory: {exp_dir}")
print(f"Config:\n{vars(args)}\n")

print(f"Loading dataset: '{args.dataset}'")
datareaders = get_data_readers(args)

model = get_model(args).to(args.device)
# TODO: take care of initializing the model if args.use_pretrained != 'none'
print(f"\nModel:\n{model}")

model = distributed_backend.transform_model(model)
group_specs = distributed_backend.get_raw_model(model).get_parameter_group_specs()
param_name_mapping = {p_name: p for p_name, p in model.named_parameters()}
optimized_params_cnt = 0

Starting Experiment: nb_debug_50M
Experiment Directory: exps/nb_debug_50M
Config:
{'config_format': 'base', 'experiment_name': 'nb_debug_50M', 'seed': 0, 'data_seed': 0, 'eval_interval': 200, 'full_eval_at': [], 'eval_batches': 32, 'device': 'cuda:0', 'distributed_backend': None, 'log_interval': 50, 'results_base_folder': './exps', 'permanent_ckpt_interval': 0, 'latest_ckpt_interval': 0, 'resume_from': None, 'resume_from_swa': None, 'auto_resume': False, 'wandb': False, 'wandb_project': 'my-project', 'wandb_run_prefix': 'none', 'eval_seq_prefix': 'none', 'log_dynamics': False, 'dynamics_logger_cfg': './src/logger/rotational_logger.yaml', 'scheduler': 'cos', 'cos_inf_steps': 0, 'iterations': 1000, 'warmup_steps': 100, 'lr': 0.0012, 'wsd_final_lr_scale': 0.0, 'wsd_fract_decay': 0.1, 'decay_type': 'linear', 'opt': 'adamw', 'batch_size': 64, 'acc_steps': 8, 'weight_decay': 0.1, 'beta1': 0.9, 'beta2': 0.95, 'grad_clip': 1.0, 'use_cage': True, 'cage_lambda': 0.5, 'cage_silence_ratio': 0.5, '

In [ ]:
for g in group_specs:
    params = []
    for p_name in g["params"]:
        translated_p_names = (
            distributed_backend.translate_model_parameter_name_for_node(p_name)
        )
        params += [param_name_mapping[p_name] for p_name in translated_p_names]
    g["params"] = params
    optimized_params_cnt += sum([p.numel() for p in g["params"]])
params_cnt = distributed_backend.get_raw_model(model).get_num_params()
nonemb_param_cnt = (
    params_cnt
    - distributed_backend.get_raw_model(model).lm_head.weight.numel()
    - distributed_backend.get_raw_model(model).transformer.wte.weight.numel()
)
print("number of parameters: %.2fM" % (params_cnt / 1e6,))
print("number of optimized parameters: %.2fM" % (optimized_params_cnt / 1e6,))
print("number of non-embedding parameters: %.2fM" % (nonemb_param_cnt / 1e6,))
    

number of parameters: 126.82M
number of optimized parameters: 126.82M
number of non-embedding parameters: 49.56M


In [ ]:
opt = torch.optim.AdamW(
    group_specs,
    lr=args.lr,
    betas=(args.beta1, args.beta2),
    weight_decay=args.weight_decay,
)

In [ ]:
print(f"\nOptimizer:\n{opt}")


Optimizer:
AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0012
    maximize: False
    weight_decay: 0.1

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0012
    maximize: False
    weight_decay: 0.0
)


In [ ]:
assert args.warmup_steps < args.iterations, "Warmup steps must be < iterations."
if args.scheduler in ["cos", "linear"]:
    # initial lr is args.lr / div_factor
    # final lr is initial_lr/final_div_factor = args.lr / div_factor / final_div_factor
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer=opt,
        max_lr=[group.get("lr", args.lr) for group in group_specs],
        total_steps=args.iterations,
        pct_start=args.warmup_steps / args.iterations,
        anneal_strategy=args.scheduler,
        cycle_momentum=False,
        div_factor=1e2,
        final_div_factor=0.1,
    )

In [ ]:
cage = CAGE(
            optimizers=[opt],
            lambda_base=args.cage_lambda,
            total_steps=args.iterations,
            silence_ratio=args.cage_silence_ratio,
            schedule=args.cage_schedule,
            track_stats=args.cage_track_stats,
        )

In [ ]:
cfg = args

In [ ]:
distributed_backend.is_master_process() and cfg.log_dynamics

False

In [ ]:
not_compiled_model = model

type_ctx = torch.amp.autocast(
    device_type="cuda",
    dtype={
        "float32": torch.float32,
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
    }[cfg.dtype],
)

curr_iter = 0


substep = curr_iter * cfg.acc_steps
train_reader, val_reader = datareaders["train"], datareaders["val"]
train_reader.set_step(substep)
stats = {"train_loss": [], "val_loss": [], "val_pp": [], "val_acc": []}
model.train()



Llama(
  (transformer): ModuleDict(
    (wte): Embedding(50304, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-6): 7 x LlamaBlock(
        (ln_1): RMSNorm()
        (attn): LlamaAttention(
          (c_attn): QuantizedLinear(
            in_features=768, out_features=2304, bias=False
            (weight_quantizer): HadamardMSEQuantizer()
            (activation_quantizer): HadamardMSEQuantizer()
          )
          (c_proj): QuantizedLinear(
            in_features=768, out_features=768, bias=False
            (weight_quantizer): HadamardMSEQuantizer()
            (activation_quantizer): HadamardMSEQuantizer()
          )
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): RMSNorm()
        (mlp): LlamaMLP(
          (w1): QuantizedLinear(
            in_features=768, out_features=2048, bias=False
            (weight_quantizer): HadamardMSEQuantizer()
            (acti

In [ ]:
from tqdm import tqdm 

pbar = tqdm(total=cfg.iterations, desc="Training Progress", position=curr_iter)

Training Progress:   0%|          | 0/1000 [00:00<?, ?it/s]

In [ ]:
ws = distributed_backend.get_world_size()
tokens = ws * substep * cfg.sequence_length * cfg.batch_size
epoch = tokens / train_reader.num_tokens

In [ ]:
import time
t_start = time.perf_counter_ns()

In [ ]:
from optim.utils import get_batch

In [ ]:
for microstep_idx in range(cfg.acc_steps):
    x, y = get_batch(train_reader, device=cfg.device)
    with type_ctx:
        with distributed_backend.get_context_for_microstep_forward(
            model=model,
            microstep_idx=microstep_idx,
            gradient_accumulation_steps=cfg.acc_steps,
        ):
            outputs = model(x, targets=y)
    loss = outputs["loss"] / cfg.acc_steps
    loss.backward()
    substep += 1


    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
opt.step()
scheduler.step()

In [ ]:
cage.step()